In [1]:
import os
import subprocess

configs_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"
python = "/home/pfuerst/venvs/NNMFit0.5_py3.11/bin/python"
script = "/home/pfuerst/venvs/NNMFit0.5_py3.11/bin/calculate_graph.py"

In [2]:
model = "FinalTopology_double_energy_length_binning"
main_out_dir = f"/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/{model}"
os.makedirs(main_out_dir, exist_ok=True)

In [12]:
DEFAULT_OVERRIDE_CONFIGS = [
    "override/livetime/hese_livetime_13yr_asr.cfg",
]
DEFAULT_OVERRIDE_COMPONENTS = [
    "override/components/astro_SPL_no_inel_no_flavor.yaml",
    "override/muon/muontemplate_hese_11features_plus_rloglmilli_econf_evtgen_bdt1_0.333333_bdt2_0.366667_scaled.yaml",
]

CONFIGS = []

In [11]:
# best fit
CONFIGS += [
    {
        "name": f"{model}",
        "analysis_config": f"analysis_configs/data/hese/debug_bdt/{model}/{model}.yaml",
        "main_config" : "main.cfg",
        "override_configs": [f"override/systematics/hese/Snowstorm_Gradients_hese_HESEBestfit_SPL/{model}_wpriors.cfg",
                             f"override/datasets_hese/data_v3/{model}.cfg"] + DEFAULT_OVERRIDE_CONFIGS,
        "override_components": DEFAULT_OVERRIDE_COMPONENTS,
        "override_parameters": None,
    },
]

In [109]:
# BDT variables
_NO_FLAVOR = ["override/components/astro_SPL_no_inel_no_flavor.yaml"] # just means no muon

_BDT_VARS = [
    ("bdt1_bdt2",          "bdt1_bdt2_10bins.cfg",                                                            DEFAULT_OVERRIDE_COMPONENTS),
    ("energy_length_analysis",          "energy_length_analysis.cfg",                                   _NO_FLAVOR),
    ("len_easym",          "Taupede_Distance_Taupede_Asymmetry_10bins.cfg",                                   _NO_FLAVOR),
    ("e1_e2",              "Taupede1_Particles_energy_Taupede2_Particles_energy_10bins.cfg",                  _NO_FLAVOR),
    ("e1_e2_zoom",              "Taupede1_Particles_energy_Taupede2_Particles_energy_10bins_zoom.cfg",                  _NO_FLAVOR),
    ("mono_energy_zenith", "MonopodFit_iMIGRAD_PPB0_energy_cscdSBU_MonopodFit4_noDC_zenith_10bins.cfg",      _NO_FLAVOR),
    ("mono_delay_qmax",    "MonopodFit_iMIGRAD_PPB0_Delay_ice_CVStatistics_q_max_doms_10bins.cfg",           _NO_FLAVOR),
    ("mono_delay_qmax_zoom",    "MonopodFit_iMIGRAD_PPB0_Delay_ice_CVStatistics_q_max_doms_10bins_zoom.cfg",           _NO_FLAVOR),
    ("vtxdist_qtot",       "cscdSBU_VertexRecoDist_CscdLLh_cscdSBU_Qtot_HLC_log_10bins.cfg",                _NO_FLAVOR),
    ("vtxdist_qtot_zoom",       "cscdSBU_VertexRecoDist_CscdLLh_cscdSBU_Qtot_HLC_log_10bins_zoom.cfg",                _NO_FLAVOR),
    ("taumono_econf",      "TauMonoDiff_rlogl_econfinement_10bins.cfg",                                       _NO_FLAVOR),
    ("taumono_econf_zoom",      "TauMonoDiff_rlogl_econfinement_10bins_zoom.cfg",                                       _NO_FLAVOR),
    ("tauspe_taumilli",    "TauSPEMilliDiff_rlogl_TauMonoMilliDiff_rlogl_10bins.cfg",                        _NO_FLAVOR),
    ("tauspe_taumilli_zoom",    "TauSPEMilliDiff_rlogl_TauMonoMilliDiff_rlogl_10bins_zoom.cfg",                        _NO_FLAVOR),
    ("evtgen_recoeratio",  "EventGeneratorDC_Thijs_length_RecoERatio_EventGeneratorDC_Max_10bins.cfg",        _NO_FLAVOR),
    ("evtgen_recoeratio_zoom",  "EventGeneratorDC_Thijs_length_RecoERatio_EventGeneratorDC_Max_10bins_zoom.cfg",        _NO_FLAVOR),

    # ("len_easym_zheyang",          "Taupede_Distance_Taupede_Asymmetry_20bins_zheyang.cfg",                                   _NO_FLAVOR),
    # ("taumono_econf_zheyang",      "TauMonoDiff_rlogl_econfinement_20bins_zheyang.cfg",                                       _NO_FLAVOR),
    # ("tauspe_taumilli_zheyang",    "TauSPEMilliDiff_rlogl_TauMonoMilliDiff_rlogl_20bins_zheyang.cfg",                        _NO_FLAVOR),
    # ("evtgen_recoeratio_zheyang",  "EventGeneratorDC_Thijs_length_RecoERatio_EventGeneratorDC_Max_20bins_zheyang.cfg",        _NO_FLAVOR),
]

_bdt_configs = []
for var, binning, components in _BDT_VARS:
    for syst_cfg, name_suffix in [
        ("override/systematics/NoSystematics_hese_combined.cfg", f"NoSystematics_{var}"),
        (f"override/systematics/hese_combined/debug_bdt/{model}/hese_combined_HESEBestfit_SPL_no_flavor_{var}.cfg", var),
    ]:
        _bdt_configs.append({
            "name": f"{model}_{name_suffix}",
            "analysis_config": f"analysis_configs/data/hese/debug_bdt/{model}/{model}_combined.yaml",
            "main_config" : "main.cfg",
            "override_configs": [
                syst_cfg,
                "override/livetime/hese_livetime_13yr_asr.cfg",
                f"override/binning/hese/combined/{binning}",
                "override/binning/hese/combined/cut_energy.cfg",
                "override/datasets_GP_globalfit/v3/globalfit_hese_clean_globalfit_v3.cfg",
            ],
            "override_components": components,
            "override_parameters": None,
        })
CONFIGS += _bdt_configs

In [13]:
# Feature name variations — fixed bdt1_bdt2 binning, varying systematics
_FEATURE_NAMES = [
    "11features_plus_rloglmilli_econf_evtgen",
    "11features",
    "11features_plus_rloglmilli",
    "11features_plus_econf",
    "11features_plus_evtgen",
]

_feature_configs = []
for feature_name in _FEATURE_NAMES:
    _components = [
        "override/components/astro_SPL_no_inel_no_flavor.yaml",
        "override/muon/muontemplate_hese_11features_plus_rloglmilli_econf_evtgen_bdt1_0.333333_bdt2_0.366667_scaled.yaml",
    ]
    for syst_cfg, name_suffix in [
        ("override/systematics/NoSystematics_hese_combined.cfg",
         f"NoSystematics_bdt1_bdt2_{feature_name}"),
        (f"override/systematics/hese_combined/debug_bdt/{feature_name}/hese_combined_HESEBestfit_SPL_no_flavor_bdt1_bdt2.cfg",
         f"bdt1_bdt2_{feature_name}"),
    ]:
        _feature_configs.append({
            "name": f"{model}_{name_suffix}",
            "analysis_config": f"analysis_configs/data/hese/debug_bdt/{model}/{model}_combined.yaml",
            "main_config": "main.cfg",
            "override_configs": [
                syst_cfg,
                "override/livetime/hese_livetime_13yr_asr.cfg",
                "override/binning/hese/combined/bdt1_bdt2_10bins.cfg",
                "override/binning/hese/combined/cut_energy.cfg",
                f"override/datasets_hese/data_v3/mcd-simpletopology_flux-hese_feat-{feature_name}/bdt1_0.333333_bdt2_0.366667_length_10.cfg",
            ],
            "override_components": _components,
            "override_parameters": None,
        })
CONFIGS += _feature_configs

In [5]:
unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
import sys
sys.path.append(unblind_script_path)
from histogram_maker import *
from do_scan_analysis import *

In [14]:
for cfg in CONFIGS:
    name = cfg["name"]
    output_dir = os.path.join(main_out_dir, name)
    os.system(f"mkdir -p {output_dir}")
    outfile = os.path.join(output_dir,"Precalculated_Graph.pickle")

    print(10*"-")
    print(name)
    print(output_dir)
    print(outfile)

    NNMFit_config_options = {
        "init_method": "from_configs",
        "main_config": os.path.join(configs_dir, cfg["main_config"]),
        "analysis_config": os.path.join(configs_dir, cfg["analysis_config"]),
        "config_dir": configs_dir,
        "override_configs": cfg["override_configs"],
        "override_components": cfg["override_components"],
        "override_parameters": cfg["override_parameters"],
    }

    cmd = [
        python, script,
        "--main_config", os.path.join(configs_dir, cfg["main_config"]),
        "--analysis_config", os.path.join(configs_dir, cfg["analysis_config"]),
        "--config_dir", configs_dir,
        "--override_configs", *cfg["override_configs"],
        "--override_components", *cfg["override_components"],
        "-o", outfile,
    ]
    if cfg["override_parameters"]:
        cmd += ["--override_parameters", *cfg["override_parameters"]]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options, out_file=f"{output_dir}/MC_Histogram.pickle")
    generate_data_hist(scan_path=output_dir)

    for component in ["Astro", "Conv", "Muon"]:
        NNMFit_config_options["analysis_config"] = os.path.join(configs_dir, cfg["analysis_config"].replace(".yaml", f"_{component}.yaml"))
        print(10*"-", "Separate component", component)
        print(NNMFit_config_options["analysis_config"])
        make_histogram_from_variables(NNMFit_config_options=NNMFit_config_options, out_file=f"{output_dir}/MC_Histogram_{component}.pickle")
    

----------
FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_rloglmilli_econf_evtgen
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_rloglmilli_econf_evtgen
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_rloglmilli_econf_evtgen/Precalculated_Graph.pickle

Called with:
main_config             : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/main.cfg
analysis_config         : /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_combined.yaml
config_dir 

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-22 03:23:46; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:23:46; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:23:46; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:23:46; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:23:46; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-22 03:23:46; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2026-06-22 03:23:46; nnm_fitter.py:92)
DEBUG:NNMFit.core.nnm_fitter: Using ve

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_rloglmilli/Data_Histogram.pickle
Full execution took 0.4247431755065918 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_rloglmilli/Data_Histogram.pickle
---------- Separate component Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_combined_Astro.yaml
---------- Separate component Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopolo

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-22 03:24:01; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:24:01; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:24:01; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:24:01; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:24:01; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-22 03:24:01; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2026-06-22 03:24:01; nnm_fitter.py:92)
DEBUG:NNMFit.core.nnm_fitter: Using ve

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_bdt1_bdt2_11features_plus_rloglmilli/Data_Histogram.pickle
Full execution took 0.4380309581756592 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_bdt1_bdt2_11features_plus_rloglmilli/Data_Histogram.pickle
---------- Separate component Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_combined_Astro.yaml
---------- Separate component Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binn

...

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_bdt1_bdt2_11features_plus_econf/Data_Histogram.pickle
Full execution took 0.4393341541290283 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_bdt1_bdt2_11features_plus_econf/Data_Histogram.pickle
---------- Separate component Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_combined_Astro.yaml
---------- Separate component Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalT

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-22 03:24:59; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:24:59; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:24:59; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:24:59; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:24:59; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-22 03:24:59; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2026-06-22 03:24:59; nnm_fitter.py:92)
DEBUG:NNMFit.core.nnm_fitter: Using ve

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_evtgen/Data_Histogram.pickle
Full execution took 0.4253695011138916 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_NoSystematics_bdt1_bdt2_11features_plus_evtgen/Data_Histogram.pickle
---------- Separate component Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_combined_Astro.yaml
---------- Separate component Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_doubl

DEBUG:NNMFit.core.loader: Loading the binning and mask for IC86_pass2_SnowStorm_FTP_HESE_Combined (2026-06-22 03:25:14; loader.py:43)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:25:14; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:25:14; rectangular_binning.py:271)
DEBUG:NNMFit.binning.rectangular_binning: binning variables: ['bdt_scores1', 'bdt_scores2'] (2026-06-22 03:25:14; rectangular_binning.py:105)
DEBUG:NNMFit.binning.rectangular_binning: Histogram of det. conf. IC86_pass2_SnowStorm_FTP_HESE_Combined has 2 dimensions  (2026-06-22 03:25:14; rectangular_binning.py:271)
INFO:NNMFit.core.nnm_fitter: Analysis type set to data (2026-06-22 03:25:14; nnm_fitter.py:87)
INFO:NNMFit.core.nnm_fitter: Loading the precalculated graph.. (2026-06-22 03:25:14; nnm_fitter.py:92)
DEBUG:NNMFit.core.nnm_fitter: Using ve

Wrote dataset to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_bdt1_bdt2_11features_plus_evtgen/Data_Histogram.pickle
Full execution took 0.4347727298736572 seconds
Loading Data Histogram: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/debug_bdt/graphs/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_bdt1_bdt2_11features_plus_evtgen/Data_Histogram.pickle
---------- Separate component Astro
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/FinalTopology_double_energy_length_binning_combined_Astro.yaml
---------- Separate component Conv
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/data/hese/debug_bdt/FinalTopology_double_energy_length_binning/Fina